In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from retail_iq.preprocessing import load_raw_data, preprocess_dates, merge_datasets, detect_outliers_iqr
from retail_iq.features import FastFeatureEngineer
from retail_iq.evaluation import evaluate_model, plot_residuals, generate_shap_summary
from retail_iq.config import MODEL_DIR, PLOT_DIR

def run_evaluation_pipeline():
    print("Loading data...")
    train, test, stores, oil, holidays, tx = load_raw_data()
    train, test, oil, holidays, tx = preprocess_dates([train, test, oil, holidays, tx])
    df = merge_datasets(train, stores, oil, holidays, tx)
    
    fe = FastFeatureEngineer(df, transactions=tx, oil_price=oil, holidays=holidays, store_meta=stores)
    fe.add_temporal_features()\
      .add_lag_and_rolling()\
      .add_onpromotion_features()\
      .add_macroeconomic_features()\
      .add_transaction_features()\
      .add_store_metadata()\
      .add_cannibalization_features()
      
    features_df = fe.transform()
    features_df = features_df.drop(columns=['sales_lag_365d'], errors='ignore')
    features_df = features_df.dropna(subset=['sales_lag_14d'])
    features_df = detect_outliers_iqr(features_df)
    
    for col in features_df.columns:
        if features_df[col].dtype == 'object' or pd.api.types.is_string_dtype(features_df[col]):
            features_df[col] = features_df[col].astype(str).astype('category').cat.codes
            
    cutoff_date = pd.Timestamp('2017-08-16')
    test_df = features_df[features_df['date'] >= cutoff_date].fillna(0)
    
    exclude_cols = ['id', 'date', 'sales', 'is_outlier']
    feature_cols = [c for c in test_df.columns if c not in exclude_cols]
    
    X_test = test_df[feature_cols].values
    y_test = test_df['sales'].values
    
    # Load Models
    try:
        xgb_model = joblib.load(MODEL_DIR / 'xgb_tuned_v1.pkl')
        lgb_model = joblib.load(MODEL_DIR / 'lgb_tuned_v1.pkl')
    except Exception as e:
        print("Models missing. Run advanced_models.py first.", e)
        return
        
    xgb_preds = np.expm1(xgb_model.predict(X_test))
    lgb_preds = np.expm1(lgb_model.predict(X_test))
    
    print("--- XGBoost Evaluation ---")
    evaluate_model(y_test, xgb_preds, "XGBoost")
    plot_residuals(y_test, xgb_preds, PLOT_DIR / 'xgb_residuals.png')
    
    print("--- LightGBM Evaluation ---")
    evaluate_model(y_test, lgb_preds, "LightGBM")
    plot_residuals(y_test, lgb_preds, PLOT_DIR / 'lgb_residuals.png')
    
    print("--- SHAP Analysis ---")
    # Using small subset of data for SHAP summary to save time in synthetic data
    X_test_df = pd.DataFrame(X_test[:50], columns=feature_cols)
    generate_shap_summary(xgb_model, X_test_df, PLOT_DIR / 'xgb_shap_summary.png')
    
    print("Evaluation completed!")

if __name__ == '__main__':
    run_evaluation_pipeline()


Loading data...


/app/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] feature_fraction is set=0.7184022681939362, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7184022681939362
--- XGBoost Evaluation ---
XGBoost: RMSLE=0.8264, RMSE=16.29, MAPE=50.78%, R²=-0.3885
Mean residual: 8.1595 (46.75% of mean actual)
--- LightGBM Evaluation ---
LightGBM: RMSLE=0.7445, RMSE=15.57, MAPE=43.17%, R²=-0.2683
Mean residual: 8.0045 (45.86% of mean actual)


--- SHAP Analysis ---


Evaluation completed!
